In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/processed/daily_raw.csv", parse_dates=["date"])
repas = pd.read_csv("../data/processed/repas_jour.csv", parse_dates=["date"])
df = df.merge(repas, on=["participant", "date"], how="left")

zones = ["zone_below", "zone_fatburn", "zone_cardio", "zone_peak"]
df["port_min"] = df[zones].sum(axis=1, min_count=1)

print(df.shape)
avant = df.isna().sum()

(456, 48)


In [2]:
cols_1_5 = ["fatigue", "mood", "sleep_quality", "soreness", "stress"]

# formulaires envoyés vides (0 partout alors que l'échelle va de 1 à 5)
vides = (df[cols_1_5] == 0).all(axis=1)
df.loc[vides, cols_1_5 + ["readiness"]] = np.nan
print(vides.sum(), "formulaires vides passés en NaN")

# readiness non exploitable pour p03 (quasi toujours 5) et p05 (0 puis 5)
df.loc[df["participant"].isin(["p03", "p05"]), "readiness"] = np.nan

2 formulaires vides passés en NaN


In [3]:
activite = ["steps", "calories", "distance_km", "hr_mean", "hr_p05", "hr_max"] + zones

partiel = (df["port_min"] < 600) | (df["steps"] == 0)
df.loc[partiel, activite] = np.nan
print(partiel.sum(), "jours de port partiel")
print(partiel.groupby(df["participant"]).sum())

28 jours de port partiel
participant
p01     0
p03    23
p05     5
dtype: int64


In [4]:
#Erreurs corrigées : 2 formulaires vides (NaN), readiness de p03 et p05 retirée , 28 jours de port partiel dont les variables d'activité passent en NaN 


In [5]:
continues = ["steps", "calories", "hr_mean", "hr_p05", "hr_max", "rhr",
             "sleep_minutes", "sleep_deep", "sleep_light", "sleep_rem", "sleep_wake",
             "overall_score", "weight"]

def z_robuste(x):
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0 or np.isnan(mad):
        return x * np.nan
    return 0.6745 * (x - med) / mad

z = df.groupby("participant")[continues].transform(z_robuste)
suspects = z.abs() > 3.5
suspects.groupby(df["participant"]).sum()

,steps,calories,hr_mean,hr_p05,hr_max,rhr,sleep_minutes,sleep_deep,sleep_light,sleep_rem,sleep_wake,overall_score,weight
participant,,,,,,,,,,,,,
p01,0,0,0,1,1,0,2,1,0,0,0,0,0
p03,0,1,0,0,0,1,3,0,1,0,1,0,0
p05,0,1,3,3,6,0,3,0,0,1,1,2,0


In [6]:
lignes = []
for col in continues:
    for i in df.index[suspects[col]]:
        lignes.append({
            "participant": df.at[i, "participant"],
            "date": df.at[i, "date"].date(),
            "variable": col,
            "valeur": df.at[i, col],
            "z": round(z.at[i, col], 1),
        })

liste = pd.DataFrame(lignes).sort_values(["participant", "date"])
liste

,participant,date,variable,valeur,z
17,p01,2019-12-02,sleep_minutes,190.000000,-3.6
18,p01,2020-01-02,sleep_minutes,510.000000,3.7
9,p01,2020-01-23,hr_max,191.000000,3.9
5,p01,2020-02-29,hr_p05,55.000000,4.0
25,p01,2020-03-03,sleep_deep,93.000000,3.5
19,p03,2019-11-01,sleep_minutes,126.000000,-3.9
16,p03,2019-12-20,rhr,64.309417,4.2
20,p03,2020-01-05,sleep_minutes,115.000000,-4.1
26,p03,2020-02-01,sleep_light,368.000000,3.6
28,p03,2020-02-03,sleep_wake,112.000000,4.5


In [7]:
df[(df["participant"] == "p03") & (df["date"] == "2020-02-06")][["steps", "calories", "port_min", "train_load", "zone_cardio", "zone_peak"]]

,steps,calories,port_min,train_load,zone_cardio,zone_peak
249,16964.0,4793.3,1225.0,0.0,15.0,24.0


In [8]:
ov = pd.read_excel("../data/Test_technique/participant-overview.xlsx", header=1)
fcmax = ov.set_index("Participant ID")["Max heart rate"]

limite = df["participant"].map(fcmax)
trop = df["hr_max"] > limite
print(df.loc[trop, ["participant", "date", "hr_max"]])
df.loc[trop, "hr_max"] = np.nan


    participant       date  hr_max
83          p01 2020-01-23   191.0
101         p01 2020-02-10   183.0
306         p05 2019-11-03   185.0
429         p05 2020-03-05   200.0


In [9]:
echelles = cols_1_5 + ["readiness"]
binaires = ["injured", "major", "alcohol"]
comptages = ["steps", "n_sessions", "n_meals", "n_photos", "glasses_of_fluid"]

for c in echelles + binaires + comptages:
    df[c] = df[c].round().astype("Int64")

df["participant"] = df["participant"].astype("category")
df.dtypes.value_counts()

float64           32
Int64             14
category           1
datetime64[us]     1
Name: count, dtype: int64

In [10]:
vars_eval = ["steps", "calories", "hr_mean", "rhr", "sleep_minutes", "overall_score",
             "fatigue", "mood", "sleep_quality", "stress"]

complet = df.dropna(subset=vars_eval)
print(len(df), "jours au total,", len(complet), "jours complets")
print(complet["participant"].value_counts())

456 jours au total, 197 jours complets
participant
p01    137
p05     60
p03      0
Name: count, dtype: int64


In [11]:
from sklearn.impute import KNNImputer

# nuits de p01 avec durée et score connus
p1 = df[df["participant"] == "p01"][["sleep_minutes", "overall_score"]].dropna().reset_index(drop=True)

# on cache 15 durées
rng = np.random.default_rng(42)
idx = rng.choice(len(p1), size=15, replace=False)
test = p1.copy()
test.loc[idx, "sleep_minutes"] = np.nan

# mediane
med = test["sleep_minutes"].fillna(test["sleep_minutes"].median())

# interpolation
interp = test["sleep_minutes"].interpolate(limit_direction="both")

# knn (on normalise sinon les minutes écrasent le score)
norm = (test - test.mean()) / test.std()
res = KNNImputer(n_neighbors=5).fit_transform(norm)
knn = pd.Series(res[:, 0] * test["sleep_minutes"].std() + test["sleep_minutes"].mean())

vrai = p1.loc[idx, "sleep_minutes"]
for nom, pred in [("mediane", med), ("interp", interp), ("knn", knn)]:
    print(nom, round((vrai - pred[idx]).abs().mean()), "min d'erreur")

mediane 34 min d'erreur
interp 50 min d'erreur
knn 35 min d'erreur


In [12]:
# Supprimer les lignes incomplètes -> que 197 jours sur 456 et plus aucun pour p03, pas possible
# Test sur 15 nuits de p01 : mediane 34 min d'erreur, knn 35, interp 50
# l'interpolation marche mal car le sommeil change beaucoup d'une nuit à l'autre
# mediane aussi bien que knn donc on garde la mediane , MICE pas testé trop lourd pour si peu de données
# On laisse les NaN dans test_dataset , la mediane sera calculée dans les modèles sur le train seulement (sinon fuite de données)
# FC de p03 et readiness de p03/p05 pas imputées

In [13]:
apres = df.isna().sum()
print(pd.DataFrame({"avant": avant, "apres": apres}).loc[lambda t: t["avant"] != t["apres"]])

df.to_csv("../data/processed/test_dataset.csv", index=False)
print(df.shape)

               avant  apres
steps             38     66
calories          38     66
distance_km       38     66
hr_mean          159    164
hr_p05           159    164
hr_max           159    168
zone_below        42     69
zone_fatburn      42     69
zone_peak         42     69
zone_cardio       42     69
fatigue          106    108
mood             106    108
readiness        106    318
sleep_quality    106    108
soreness         106    108
stress           106    108
(456, 48)


In [14]:
# avant/après : +28 sur l'activité (jours de port partiel), +5 sur la FC, +4 en plus sur hr_max (au dessus FC max)
# +2 sur le questionnaire (formulaires vides), readiness 106 -> 318 car retirée pour p03 et p05